<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: C1-Semi/unstructured Analytics Tutorial Exercises (Part B)

For this tutorial, we will continue the same scenario and process as Part A of the tutorial notebook (B3). Recall that we used "Brisbane 2032 Olympics" as a suggested topic and completed the following process:
1. Use the Guardian API to undertake your own search and obtain a json file of documents
2. Create a TF/IDF document-term matrix for your documents

For this tutorial session, we are going to extend our process to include topic modelling.

In [1]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import pandas as pd
import json
import random

### Question

What is the question you investigated in B3, or what question would you like to investigate: 

**Question**: [Write your question here and explain why it's significant.] 

### B3 recap and set up for new analyses

We need to load the articles we previously collected, and extend our dataframe to include `lda` and `nmf` for comparison.

To save time, we will reuse code from B3 tutorial directly. For explanations and notes, refer back to B3 tutorial notebook. 

#### Load data and find relevant articles

In [2]:
# Load the data - articles saved from B3 tutorial
file_path = "data/"
file_name = "brisbane_olympics_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Loaded 126 articles from brisbane_olympics_articles.json


In [3]:
# process articles to find relevant articles for the topic
## You might need to customise this part if you use a different topic 

# e.g., remove articles contains 'as it happened' - why?
## get a list of article titles 
titles = list(articles.keys())
## create an empty list to store filtered titles
filtered_titles = []

for title in titles:
    if "as it happens" not in title:
        filtered_titles.append(title)
        
# e.g., include titles that contain 'Brisbane' and 'Olympic' - why?
filtered_titles_2 = [title for title in filtered_titles if 'Brisbane' in title and 'Olympic' in title]

# Filter the JSON data to only include these titles
articles_filtered = {title: content for title, content in articles.items() if title in filtered_titles_2}
len(articles_filtered)

13

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. This time we will include `lda` and `nmf` in the dataframe. 

In [4]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=articles_filtered.keys(),columns=['count', 'tfidf', 'lda', 'nmf'])
terms_df

,count,tfidf,lda,nmf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]",NaN,NaN,NaN,NaN
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]",NaN,NaN,NaN,NaN
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],NaN,NaN,NaN,NaN
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],NaN,NaN,NaN,NaN
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],NaN,NaN,NaN,NaN
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],NaN,NaN,NaN,NaN
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],NaN,NaN,NaN,NaN
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],NaN,NaN,NaN,NaN
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],NaN,NaN,NaN,NaN
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],NaN,NaN,NaN,NaN


#### Previous analyses for term frequency and TFIDF

In [5]:
# term frequency
# Set parameters appropriate to your data
count_vectorizer = CountVectorizer(max_df=0.80,min_df=2,max_features=10000,stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles_filtered.values())
# Get the terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), index = terms_df.index, columns=feature_names)
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

# tfidf
# Set parameters appropriate to your data
tfidf_vectorizer = TfidfVectorizer(max_df=0.80, min_df=2, max_features=10000, stop_words="english")
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles_filtered.values())
# list of feature names
feature_names = tfidf_vectorizer.get_feature_names_out()
# create a df to combine matrix with feature names
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), index=articles_filtered.keys(), columns=feature_names)
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    terms_df.at[idx,'tfidf'] = list(tfidf.keys())

terms_df


,count,tfidf,lda,nmf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]","[words, says, believe, search, passion, video,...","[words, believe, says, advertising, passion, s...",NaN,NaN
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]","[says, story, people, history, men, today, com...","[says, story, men, history, jones, england, to...",NaN,NaN
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],"[board, women, committee, planning, indigenous...","[board, women, directors, committee, planning,...",NaN,NaN
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],"[park, aboriginal, stadium, heritage, victoria...","[park, aboriginal, heritage, federal, victoria...",NaN,NaN
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],"[arbib, aoc, sports, executive, athletes, comm...","[arbib, aoc, sports, executive, experience, at...",NaN,NaN
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],"[crocodile, rowing, crocodiles, river, world, ...","[crocodile, rowing, crocodiles, river, behavio...",NaN,NaN
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],"[new, park, site, victoria, bleijie, stadium, ...","[bleijie, new, victoria, park, site, king, dea...",NaN,NaN
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],"[park, victoria, stadium, people, plan, save, ...","[park, victoria, stadium, save, people, newman...",NaN,NaN
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],"[police, people, australia, day, sydney, nsw, ...","[police, people, australia, sydney, nsw, man, ...",NaN,NaN
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],"[new, stadium, centre, build, crisafulli, priv...","[new, private, sector, build, stadium, arena, ...",NaN,NaN


### Topic modelling with Latent Dirichlet Allocation (LDA)
We will follow the same process from the lecture to extract top 10 terms using LDA. It's important to refer to the lecture notebook to understand how topic modelling work. Explore different parameters.

In [6]:
# Set number of topics
num_topics = 15
# Set max number of iteractions
max_iterations = 20

# Create the model
lda_model = LatentDirichletAllocation(n_components=num_topics,max_iter=max_iterations,learning_method='online')

# Fit the model to the data, and use the model to transform the data (do the decomposition)
doc_topic_matrix = lda_model.fit_transform(count_dt_matrix)

# Obtain the topics
topic_term_matrix = lda_model.components_

#### View the topics

In [7]:
# Get the topics and their terms
lda_topic_dict = {}
for index, topic in enumerate(topic_term_matrix):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    lda_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in lda_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'stadium': np.float64(37.2167), 'park': np.float64(37.1853), 'new': np.float64(33.6081), 'victoria': np.float64(30.7414), 'plan': np.float64(19.5789), 'city': np.float64(18.6904), 'crisafulli': np.float64(16.7827), 'site': np.float64(15.9143), 'build': np.float64(15.8949), 'people': np.float64(15.0665)}

topic_1
{'new': np.float64(0.2533), 'stadium': np.float64(0.2174), 'park': np.float64(0.2161), 'venues': np.float64(0.2099), 'victoria': np.float64(0.1885), 'says': np.float64(0.1884), 'plan': np.float64(0.1876), 'centre': np.float64(0.1861), 'crisafulli': np.float64(0.1846), 'planning': np.float64(0.1827)}

topic_2
{'police': np.float64(0.1728), 'crocodile': np.float64(0.1623), 'australia': np.float64(0.1609), 'sydney': np.float64(0.1606), 'just': np.float64(0.1549), 'local': np.float64(0.1545), 'world': np.float64(0.153), 'people': np.float64(0.1524), 'did': np.float64(0.1514), 'regional': np.float64(0.1511)}

topic_3
{'police': np.float64(72.932), 'people': np.float64(66.45

##### Discussions

1. What kinds of 'themes' can we infer from each topic? Can we get a sense what most of the articles focus on? 
- [Jot down your notes here]

2. How might we be able to use insights from the topics to address your question?
- [Jot down your notes here]

#### Update the terms matrix

> Note: the code below is slightly different from the lecture, as in our previous practice we set the article names as the dataframe index instead of using the default index.

In [8]:
for article_name,topic in zip(terms_df.index, doc_topic_matrix):
    topic_num = topic.argmax() # which item (the location) has the max value
    top_topic = lda_topic_dict[f"topic_{topic_num}"]
    terms_df.at[article_name,'lda'] = list(top_topic.keys())

terms_df

,count,tfidf,lda,nmf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]","[words, says, believe, search, passion, video,...","[words, believe, says, advertising, passion, s...","[says, words, people, story, believe, world, t...",NaN
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]","[says, story, people, history, men, today, com...","[says, story, men, history, jones, england, to...","[says, words, people, story, believe, world, t...",NaN
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],"[board, women, committee, planning, indigenous...","[board, women, directors, committee, planning,...","[arbib, women, committee, aoc, sports, executi...",NaN
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],"[park, aboriginal, stadium, heritage, victoria...","[park, aboriginal, heritage, federal, victoria...","[stadium, park, new, victoria, plan, city, cri...",NaN
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],"[arbib, aoc, sports, executive, athletes, comm...","[arbib, aoc, sports, executive, experience, at...","[arbib, women, committee, aoc, sports, executi...",NaN
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],"[crocodile, rowing, crocodiles, river, world, ...","[crocodile, rowing, crocodiles, river, behavio...","[crocodile, rowing, crocodiles, river, world, ...",NaN
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],"[new, park, site, victoria, bleijie, stadium, ...","[bleijie, new, victoria, park, site, king, dea...","[stadium, park, new, victoria, plan, city, cri...",NaN
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],"[park, victoria, stadium, people, plan, save, ...","[park, victoria, stadium, save, people, newman...","[stadium, park, new, victoria, plan, city, cri...",NaN
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],"[police, people, australia, day, sydney, nsw, ...","[police, people, australia, sydney, nsw, man, ...","[police, people, australia, day, nsw, sydney, ...",NaN
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],"[new, stadium, centre, build, crisafulli, priv...","[new, private, sector, build, stadium, arena, ...","[stadium, park, new, victoria, plan, city, cri...",NaN


### Topic modelling with Non-negative Matrix Factorisation (NMF)


[NMF](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is a different algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It also factorises the document-term matrix into 2 factor matrices: document-topic and topic-term.

In [9]:
# Set the number of topics
num_topics = 15

# Create the model
nmf_model = NMF(n_components=num_topics, init='random', beta_loss='frobenius')

# Fit the model to the data and use it to transform the data
doc_topic_nmf = nmf_model.fit_transform(tfidf_dt_matrix)

topic_term_nmf = nmf_model.components_

In [10]:
# Get the topics and their terms
nmf_topic_dict = {}
for index, topic in enumerate(topic_term_nmf):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    nmf_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in nmf_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'words': np.float64(2.5287), 'believe': np.float64(1.5804), 'says': np.float64(1.264), 'advertising': np.float64(0.9483), 'passion': np.float64(0.9483), 'search': np.float64(0.9483), 'video': np.float64(0.9483), 'vision': np.float64(0.8409), 'west': np.float64(0.6322), 'dream': np.float64(0.6322)}

topic_1
{'stadium': np.float64(0.5018), 'time': np.float64(0.4631), 'planning': np.float64(0.4435), 'crisafulli': np.float64(0.4392), 'venues': np.float64(0.4272), 'build': np.float64(0.4103), 'victoria': np.float64(0.4044), 'plans': np.float64(0.3495), 'plan': np.float64(0.3495), 'significant': np.float64(0.314)}

topic_2
{'procurement': np.float64(0.9575), 'review': np.float64(0.8305), 'special': np.float64(0.766), 'required': np.float64(0.7648), 'processes': np.float64(0.6797), 'case': np.float64(0.6108), 'projects': np.float64(0.6107), 'park': np.float64(0.5426), 'including': np.float64(0.4706), 'cost': np.float64(0.4582)}

topic_3
{'police': np.float64(3.0025), 'people': np.flo

##### Discussions
1. What kinds of 'themes' can we infer from each topic? Can we get a sense what most of the articles focus on? Any similarities/differences from the LDA results?
- [Jot down your notes here]

2. How might we be able to use insights from the topics to address your question?
- [Jot down your notes here]

#### Update the terms matrix

In [11]:
for article_name,topic in zip(terms_df.index, doc_topic_nmf):
    topic_num = topic.argmax()
    top_topic = nmf_topic_dict[f"topic_{topic_num}"]
    terms_df.at[article_name,'nmf'] = list(top_topic.keys())

terms_df

,count,tfidf,lda,nmf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]","[words, says, believe, search, passion, video,...","[words, believe, says, advertising, passion, s...","[says, words, people, story, believe, world, t...","[words, believe, says, advertising, passion, s..."
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]","[says, story, people, history, men, today, com...","[says, story, men, history, jones, england, to...","[says, words, people, story, believe, world, t...","[says, story, men, history, england, jones, to..."
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],"[board, women, committee, planning, indigenous...","[board, women, directors, committee, planning,...","[arbib, women, committee, aoc, sports, executi...","[make, board, aboriginal, policy, indigenous, ..."
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],"[park, aboriginal, stadium, heritage, victoria...","[park, aboriginal, heritage, federal, victoria...","[stadium, park, new, victoria, plan, city, cri...","[park, aboriginal, victoria, heritage, federal..."
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],"[arbib, aoc, sports, executive, athletes, comm...","[arbib, aoc, sports, executive, experience, at...","[arbib, women, committee, aoc, sports, executi...","[arbib, aoc, sports, executive, experience, at..."
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],"[crocodile, rowing, crocodiles, river, world, ...","[crocodile, rowing, crocodiles, river, behavio...","[crocodile, rowing, crocodiles, river, world, ...","[crocodile, rowing, crocodiles, river, behavio..."
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],"[new, park, site, victoria, bleijie, stadium, ...","[bleijie, new, victoria, park, site, king, dea...","[stadium, park, new, victoria, plan, city, cri...","[bleijie, new, victoria, park, site, king, dea..."
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],"[park, victoria, stadium, people, plan, save, ...","[park, victoria, stadium, save, people, newman...","[stadium, park, new, victoria, plan, city, cri...","[park, victoria, people, save, stadium, newman..."
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],"[police, people, australia, day, sydney, nsw, ...","[police, people, australia, sydney, nsw, man, ...","[police, people, australia, day, nsw, sydney, ...","[police, people, australia, nsw, sydney, man, ..."
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],"[new, stadium, centre, build, crisafulli, priv...","[new, private, sector, build, stadium, arena, ...","[stadium, park, new, victoria, plan, city, cri...","[new, private, sector, build, stadium, arena, ..."


### Check against articles

In [12]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc.index}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print("\t>> NMF:\t\t",doc['nmf'])
    print()

[3] Index(['count', 'tfidf', 'lda', 'nmf'], dtype='str')
	>> Counts:	 ['park', 'aboriginal', 'stadium', 'heritage', 'victoria', 'federal', 'plan', 'city', 'people', 'act']
	>> TFIDF:	 ['park', 'aboriginal', 'heritage', 'federal', 'victoria', 'stadium', 'protection', 'consultation', 'plan', 'nations']
	>> LDA:		 ['stadium', 'park', 'new', 'victoria', 'plan', 'city', 'crisafulli', 'site', 'build', 'people']
	>> NMF:		 ['park', 'aboriginal', 'victoria', 'heritage', 'federal', 'stadium', 'protection', 'consultation', 'plan', 'nations']

[9] Index(['count', 'tfidf', 'lda', 'nmf'], dtype='str')
	>> Counts:	 ['new', 'stadium', 'centre', 'build', 'crisafulli', 'private', 'arena', '000', 'sector', 'plan']
	>> TFIDF:	 ['new', 'private', 'sector', 'build', 'stadium', 'arena', 'centre', 'crisafulli', 'choice', '000']
	>> LDA:		 ['stadium', 'park', 'new', 'victoria', 'plan', 'city', 'crisafulli', 'site', 'build', 'people']
	>> NMF:		 ['new', 'private', 'sector', 'build', 'stadium', 'arena', 'centre

**What can you find from the comparison? Is it helpful to address your question? If not, what can you get out of it and use it for further investigation?**

[Write your thoughts here]


In [13]:
# Continue your investigations - add more cells


## Refine your analysis

Once you have worked through the process. Try tweaking the parameters to obtain better results for your data.

#### Advanced

You may obtain better results by doing the following:

- Creating smaller documents (e.g. article paragraphs)
- Pre-processing the text by Stemming or Lemmatizing, and by removing additional stop words.
- ???